In [ ]:
# ============================================================
# FULL CODE - FACE MATCHING DEEP LEARNING (InsightFace ArcFace)
# Stack: RetinaFace (detect) + ArcFace buffalo_l (embedding)
# Môi trường: Google Colab Free T4 GPU
# ============================================================


# ════════════════════════════════════════════════════════════
# CELL 1 — Cài đặt thư viện
# ════════════════════════════════════════════════════════════

!pip install -q insightface onnxruntime-gpu
!pip install -q opencv-python-headless tqdm Pillow

import torch
print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Không có GPU!"}')
print('✅ Cài đặt hoàn tất!')


# ════════════════════════════════════════════════════════════
# CELL 2 — Kết nối Drive + Upload ảnh mẫu + Khởi tạo model
# ════════════════════════════════════════════════════════════

from google.colab import drive, files, auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import cv2, io, time, shutil, json
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import insightface
from insightface.app import FaceAnalysis

# Mount Drive
drive.mount('/content/drive')
print('✅ Google Drive đã kết nối!')

# Xác thực Drive API
auth.authenticate_user()
creds, _ = default()
service = build('drive', 'v3', credentials=creds)
print('✅ Drive API sẵn sàng!')

# ── Khởi tạo InsightFace (RetinaFace + ArcFace buffalo_l) ────
#    buffalo_l = model lớn, cân bằng tốc độ/chính xác trên T4
print('\n⏳ Đang load model InsightFace (buffalo_l)...')
app = FaceAnalysis(
    name='buffalo_l',           # RetinaFace detect + ArcFace embed
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
app.prepare(ctx_id=0, det_size=(640, 640))
print('✅ Model sẵn sàng! (RetinaFace + ArcFace)')

# ── Upload ảnh mẫu khuôn mặt ─────────────────────────────────
REFERENCE_DIR = Path('/content/reference_faces')
REFERENCE_DIR.mkdir(exist_ok=True)

print('\n📤 Upload 3–5 ảnh mẫu khuôn mặt của bạn (ảnh rõ mặt, nhìn thẳng):')
uploaded = files.upload()

reference_embeddings = []   # list of 512-dim ArcFace vectors
for filename, content in uploaded.items():
    save_path = REFERENCE_DIR / filename
    save_path.write_bytes(content)

    img_bgr = cv2.imdecode(np.frombuffer(content, np.uint8), cv2.IMREAD_COLOR)
    faces   = app.get(img_bgr)

    if faces:
        # Lấy khuôn mặt lớn nhất (diện tích bbox) nếu có nhiều mặt
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
        reference_embeddings.append(face.normed_embedding)   # vector 512-d, đã normalize
        print(f'  ✅ {filename} — det_score={face.det_score:.3f}')
    else:
        print(f'  ❌ {filename} — Không phát hiện được mặt, thử ảnh khác')

# Tính embedding trung bình làm reference (robust hơn dùng 1 ảnh)
if reference_embeddings:
    mean_embedding = np.mean(reference_embeddings, axis=0)
    mean_embedding /= np.linalg.norm(mean_embedding)   # re-normalize
    print(f'\n🎯 Đã tạo reference embedding từ {len(reference_embeddings)} ảnh. Sẵn sàng!')
else:
    print('\n❌ Không có ảnh mẫu hợp lệ. Hãy upload lại ảnh rõ mặt hơn.')


# ════════════════════════════════════════════════════════════
# CELL 3 — Cấu hình (chỉnh sửa theo nhu cầu)
# ════════════════════════════════════════════════════════════
# Sample
SHARED_FOLDER_IDS = [
    '1fHFqQGYBtEeGr_jKh7n_ehwMDzfdiiWx',
]

# ── Ngưỡng cosine similarity (ArcFace dùng similarity, KHÁC với face_recognition) ──
# ArcFace embedding đã normalize → dot product = cosine similarity
# 0.0 = hoàn toàn khác | 1.0 = giống hệt
# Khuyến nghị: 0.35–0.45 (chặt hơn so với tolerance của face_recognition)
SIMILARITY_THRESHOLD = 0.40   # tăng = lỏng hơn | giảm = chặt hơn

BATCH_SIZE = 200
IMG_EXTS   = {'.jpg', '.jpeg', '.png', '.webp', '.gif', '.bmp', '.tiff'}

DOWNLOAD_DIR = Path('/content/_batch_tmp')
OUTPUT_DIR   = Path('/content/drive/MyDrive/matched_photos_arcface')

DOWNLOAD_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Cấu hình hoàn tất!')
print(f'📁 Kết quả: {OUTPUT_DIR}')
print(f'🎯 Threshold cosine similarity: {SIMILARITY_THRESHOLD}')
print(f'📦 Batch size: {BATCH_SIZE} ảnh/lần')


# ════════════════════════════════════════════════════════════
# CELL 4 — Hàm tiện ích (không cần sửa)
# ════════════════════════════════════════════════════════════

def list_all_images(folder_id, service):
    """Đệ quy lấy (file_id, file_name) — KHÔNG tải file."""
    items, page_token = [], None
    while True:
        try:
            resp = service.files().list(
                q=f"'{folder_id}' in parents and trashed=false",
                fields='nextPageToken, files(id, name, mimeType)',
                pageToken=page_token,
                pageSize=1000,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()
        except Exception as e:
            print(f'⚠️ Không list được {folder_id}: {e}')
            break

        for f in resp.get('files', []):
            if f['mimeType'] == 'application/vnd.google-apps.folder':
                items.extend(list_all_images(f['id'], service))
            elif Path(f['name']).suffix.lower() in IMG_EXTS:
                items.append((f['id'], f['name']))

        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return items


def download_file(file_id, dest_path, service):
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        req = service.files().get_media(fileId=file_id, supportsAllDrives=True)
        with open(dest_path, 'wb') as fh:
            dl = MediaIoBaseDownload(fh, req, chunksize=4 * 1024 * 1024)
            done = False
            while not done:
                _, done = dl.next_chunk()
        return True
    except Exception:
        return False


def check_face_match(image_path, ref_embedding, threshold, app):
    """
    RetinaFace detect → ArcFace embed → cosine similarity với ref_embedding.
    Trả về (True, best_similarity) nếu khớp, (False, None) nếu không.
    """
    try:
        img_bgr = cv2.imread(str(image_path))
        if img_bgr is None:
            return False, None

        # Resize ảnh lớn để tăng tốc (RetinaFace vẫn detect tốt ở 1080p)
        h, w = img_bgr.shape[:2]
        if max(h, w) > 1920:
            scale = 1920 / max(h, w)
            img_bgr = cv2.resize(img_bgr, (int(w * scale), int(h * scale)))

        faces = app.get(img_bgr)
        if not faces:
            return False, None

        best_sim = -1.0
        for face in faces:
            # Bỏ qua khuôn mặt quá nhỏ (< 20px) — thường là noise
            bw = face.bbox[2] - face.bbox[0]
            bh = face.bbox[3] - face.bbox[1]
            if bw < 20 or bh < 20:
                continue

            # Cosine similarity (cả hai đã normalize → dot product)
            sim = float(np.dot(face.normed_embedding, ref_embedding))
            if sim > best_sim:
                best_sim = sim

        if best_sim >= threshold:
            return True, best_sim

    except Exception:
        pass

    return False, None


print('✅ Hàm tiện ích sẵn sàng!')


# ════════════════════════════════════════════════════════════
# CELL 5 — CHẠY CHÍNH: quét metadata + vòng for batch
# ════════════════════════════════════════════════════════════

# ── A: Quét metadata (1 lần, không tải ảnh) ─────────────────

print('🔍 Đang quét metadata toàn bộ folder...\n')
all_records = []

for folder_id in SHARED_FOLDER_IDS:
    print(f'  📂 {folder_id}')
    records = list_all_images(folder_id, service)
    print(f'     → {len(records)} ảnh')
    all_records.extend(records)

total    = len(all_records)
n_batches = -(-total // BATCH_SIZE)
print(f'\n📊 Tổng: {total} ảnh | {n_batches} batch × {BATCH_SIZE} ảnh\n')


# ── B: Vòng for xử lý batch ──────────────────────────────────

results_log = []
saved_count = 0
failed_dl   = []
start_time  = time.time()

for batch_idx in range(n_batches):
    batch = all_records[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]

    print(f'\n{"─"*55}')
    print(f'📦  BATCH {batch_idx+1}/{n_batches}  ({len(batch)} ảnh)')
    print(f'{"─"*55}')

    # 1. Tải batch
    batch_local = []
    for file_id, file_name in tqdm(batch, desc='⬇️  Tải', unit='ảnh'):
        dest = DOWNLOAD_DIR / file_name
        if dest.exists():
            dest = DOWNLOAD_DIR / f'{dest.stem}_{file_id[:6]}{dest.suffix}'
        if download_file(file_id, dest, service):
            batch_local.append(dest)
        else:
            failed_dl.append(file_name)

    print(f'     ✅ Tải OK: {len(batch_local)} | ❌ Lỗi: {len(batch)-len(batch_local)}')

    # 2. Face detect + ArcFace matching → lưu ngay lên Drive
    batch_matched = 0
    for img_path in tqdm(batch_local, desc='🔍 ArcFace matching', unit='ảnh'):
        is_match, sim = check_face_match(img_path, mean_embedding, SIMILARITY_THRESHOLD, app)
        if is_match:
            out = OUTPUT_DIR / img_path.name
            if out.exists():
                out = OUTPUT_DIR / f'{img_path.stem}_{saved_count}{img_path.suffix}'
            try:
                shutil.copy2(img_path, out)
                saved_count   += 1
                batch_matched += 1
                results_log.append({
                    'file'      : img_path.name,
                    'similarity': round(sim, 4),
                    'confidence': f'{sim * 100:.1f}%',
                    'batch'     : batch_idx + 1,
                })
            except Exception as e:
                print(f'  ⚠️ Không lưu được {img_path.name}: {e}')

    print(f'     🎉 Khớp: {batch_matched} | Tổng đã lưu: {saved_count}')

    # 3. Xóa batch tạm
    for p in batch_local:
        try: p.unlink()
        except: pass

    # 4. Thống kê
    free_gb = shutil.disk_usage('/content').free / 1e9
    elapsed = (time.time() - start_time) / 60
    eta     = (elapsed / (batch_idx + 1)) * (n_batches - batch_idx - 1)
    print(f'     💾 Ổ cứng còn: {free_gb:.1f} GB | ⏱️ Đã: {elapsed:.1f} phút | ETA: {eta:.1f} phút')

    # 5. Lưu log tạm phòng crash
    log_path = OUTPUT_DIR / 'results_log.json'
    log_path.write_text(json.dumps(results_log, ensure_ascii=False, indent=2))


# ── Kết quả cuối ─────────────────────────────────────────────

total_min = (time.time() - start_time) / 60
print(f'\n{"═"*55}')
print(f'🏁  HOÀN TẤT!')
print(f'{"═"*55}')
print(f'✅  Đã quét  : {total} ảnh trong {total_min:.1f} phút')
print(f'🎉  Tìm thấy : {saved_count} ảnh có khuôn mặt của bạn')
print(f'📁  Lưu tại  : {OUTPUT_DIR}')
if failed_dl:
    print(f'⚠️  Lỗi tải  : {len(failed_dl)} file')


# ════════════════════════════════════════════════════════════
# CELL 6 — Preview kết quả + phân tích similarity distribution
# ════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

if not results_log:
    print('😔 Không tìm thấy ảnh nào.')
    print('💡 Thử tăng SIMILARITY_THRESHOLD lên 0.35 rồi chạy lại Cell 5.')
else:
    # ── Biểu đồ phân phối similarity ─────────────────────────
    sims = [r['similarity'] for r in results_log]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(sims, bins=20, color='#2196F3', edgecolor='white', linewidth=0.5)
    axes[0].axvline(SIMILARITY_THRESHOLD, color='red', linestyle='--', label=f'Threshold={SIMILARITY_THRESHOLD}')
    axes[0].set_title('Phân phối Cosine Similarity', fontweight='bold')
    axes[0].set_xlabel('Similarity')
    axes[0].set_ylabel('Số ảnh')
    axes[0].legend()

    # Top 10 ảnh khớp nhất
    top10 = sorted(results_log, key=lambda x: -x['similarity'])[:10]
    names = [r['file'][:15] for r in top10]
    vals  = [r['similarity'] for r in top10]
    axes[1].barh(names[::-1], vals[::-1], color='#4CAF50')
    axes[1].axvline(SIMILARITY_THRESHOLD, color='red', linestyle='--')
    axes[1].set_title('Top 10 ảnh khớp nhất', fontweight='bold')
    axes[1].set_xlabel('Similarity')
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'similarity_analysis.png'), dpi=100)
    plt.show()

    # ── Preview 16 ảnh ───────────────────────────────────────
    matched_files = sorted(OUTPUT_DIR.glob('*.[jp][pn]g'),
                           key=lambda p: p.stat().st_mtime)[:16]
    cols = 4
    rows = -(-len(matched_files) // cols)
    fig, axes2 = plt.subplots(rows, cols, figsize=(16, rows * 4))
    axes2 = np.array(axes2).flatten()

    for i, img_path in enumerate(matched_files):
        try:
            entry = next((r for r in results_log if r['file'] == img_path.name), None)
            sim_str = f"sim={entry['similarity']:.3f}" if entry else ''
            color   = 'green' if entry and entry['similarity'] >= 0.45 else 'orange'
            axes2[i].imshow(mpimg.imread(str(img_path)))
            axes2[i].set_title(f'{img_path.name[:18]}\n{sim_str}', fontsize=8, color=color)
            axes2[i].axis('off')
        except:
            axes2[i].axis('off')

    for j in range(len(matched_files), len(axes2)):
        axes2[j].axis('off')

    plt.suptitle(f'Tìm thấy {saved_count} ảnh — Preview {len(matched_files)} ảnh',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'preview.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print(f'\n💡 Xem toàn bộ {saved_count} ảnh tại Google Drive: matched_photos_arcface/')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 13.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 92.7 MB/s eta 0:00:00
✅ GPU: Tesla T4
✅ Cài đặt hoàn tất!
Mounted at /content/drive
✅ Google Drive đã kết nối!
✅ Drive API sẵn sàng!

⏳ Đang load model InsightFace (buffalo_l)...
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 105867.37KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

Saving RedsMotion-04982.jpg to RedsMotion-04982.jpg
Saving RedsMotion-04981.jpg to RedsMotion-04981.jpg
Saving RedsMotion-02350.jpg to RedsMotion-02350.jpg
Saving RedsMotion-02349.jpg to RedsMotion-02349.jpg
  ✅ RedsMotion-04982.jpg — det_score=0.909
  ✅ RedsMotion-04981.jpg — det_score=0.912
  ✅ RedsMotion-02350.jpg — det_score=0.902
  ✅ RedsMotion-02349.jpg — det_score=0.912

🎯 Đã tạo reference embedding từ 4 ảnh. Sẵn sàng!


✅ Cấu hình hoàn tất!
📁 Kết quả: /content/drive/MyDrive/matched_photos_arcface
🎯 Threshold cosine similarity: 0.4
📦 Batch size: 200 ảnh/lần
✅ Hàm tiện ích sẵn sàng!
🔍 Đang quét metadata toàn bộ folder...

  📂 1fHFqQGYBtEeGr_jKh7n_ehwMDzfdjiWx
     → 3042 ảnh
  📂 1QLeiexgxxkIouwns-j2z14t_PFArJANv
     → 253 ảnh
  📂 1x7vtqcCv2EQey0L8Gc5f7XKZ3HbkxU2X
     → 7881 ảnh
  📂 1F7NCkXFOulReYuhGD6MF7srPOWwYDSdp
     → 325 ảnh

📊 Tổng: 11501 ảnh | 58 batch × 200 ảnh


───────────────────────────────────────────────────────
📦  BATCH 1/58  (200 ảnh)
───────────────────────────────────────────────────────


...

⬇️  Tải:   0%|          | 0/200 [00:00<?, ?ảnh/s]

     ✅ Tải OK: 200 | ❌ Lỗi: 0


🔍 ArcFace matching:   0%|          | 0/200 [00:00<?, ?ảnh/s]

     🎉 Khớp: 0 | Tổng đã lưu: 34
     💾 Ổ cứng còn: 73.1 GB | ⏱️ Đã: 205.0 phút | ETA: 71.5 phút

───────────────────────────────────────────────────────
📦  BATCH 44/58  (200 ảnh)
───────────────────────────────────────────────────────


⬇️  Tải:   0%|          | 0/200 [00:00<?, ?ảnh/s]